# Case study: what drives the nightly price of a Berlin listing? And which customers churn?

*Session 6 · Introduction to machine learning with regression · Author: course team · Licence: CC-BY-4.0*

The notebook follows the three practice tasks of the session:

1. **Block 1**: map a price model for Berlin hosts to the stages of the lifecycle; split the listings; fit and
   interpret a linear regression of the (log) nightly price; evaluate it in euros against a baseline.
2. **Block 2**: training versus test error for increasing model complexity; least squares versus Huber and
   quantile regression for a price with extreme values (median and 90th-percentile price for a host).
3. **Block 3**: logistic regression for churn of the IBM Telco customers (unchanged from the module plan).

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared with
`uv run python case-study/prepare_airbnb.py`; collected from public listing pages, host names and texts removed.
As in Sessions 4 and 5, prices are analysed for listings with a minimum stay below 28 nights, because the price
field of medium-term listings is not comparable. The Telco data are downloaded from GitHub in block 3.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import HuberRegressor, LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, mean_absolute_error, r2_score,
                             root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data" / "airbnb"
assert (DATA / "listings.parquet").exists(), "run: uv run python case-study/prepare_airbnb.py"

listings = pd.read_parquet(DATA / "listings.parquet")
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()
short["log_price"] = np.log(short["price"])
short["km_to_centre"] = 111.2 * np.hypot(short["latitude"] - 52.5219,                       # Alexanderplatz
                                         (short["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
print(short.shape)
short[["price", "accommodates", "bedrooms", "km_to_centre"]].describe().round(1)

## Block 1 · Lifecycle, split and linear regression

### 1.1 A pricing aid for hosts along the lifecycle

**Exercise.** Complete the table. Think about what is known when a host lists a *new* flat.

| Step | What it means for the price model | Session |
|---|---|---|
| 01 Problem definition | target = price per night (log scale); metric = MAE in EUR; baseline = median price of the training listings | S6 |
| 02 Data collection | … (public snapshot scraped by Inside Airbnb; which columns would a new host have? `estimated_revenue_l365d` is computed from the price) | S3, S9 |
| 03 Cleaning | … (medium-term listings, extreme prices, missing bedrooms) | S4 |
| 04 EDA | … | S5 |
| 05 Features | … | S9 |
| 06 Model selection | … | S7, S10 |
| 07 Training | … | S6 |
| 08 Evaluation and tuning | … (hosts with many similar listings) | S6, S7 |
| 09 Deployment | … | S16 |
| 10 Monitoring | … (the next snapshot; new rules for registration; seasons) | S16 |

### 1.2 Train–test split and baseline

In [ ]:
train, test = train_test_split(short, test_size=0.2, random_state=42)
baseline = train["price"].median()
print(len(train), len(test), "baseline (median price):", baseline)
print("baseline MAE on test:", round(mean_absolute_error(test["price"], np.full(len(test), baseline)), 1))

### 1.3 Simple and multiple linear regression

The target is the log price: prices are right-skewed (Session 5), and on the log scale a coefficient reads as a
percentage difference (e^b − 1).

In [ ]:
simple = smf.ols("log_price ~ accommodates", data=train).fit()
multi = smf.ols('log_price ~ accommodates + C(room_type) + km_to_centre + C(district, Treatment("Mitte"))',
                data=train).fit()
print(simple.params.round(3))
table = pd.DataFrame({"coef": multi.params, "percent": (np.exp(multi.params) - 1) * 100,
                      "ci_low": multi.conf_int()[0], "ci_high": multi.conf_int()[1]})
print("R2 simple:", round(simple.rsquared, 3), " R2 multiple:", round(multi.rsquared, 3))
table.round(3)

**Exercise.** Interpret the coefficients of `accommodates`, of a private room and of Neukölln in one sentence
each ("holding the other features fixed, …"). Why is the guest coefficient smaller in the multiple model than in
the simple one?

### 1.4 Metrics in euros, compared with the baseline

In [ ]:
X = pd.get_dummies(short[["accommodates", "room_type", "km_to_centre", "district"]], drop_first=True, dtype=float)
X_train, X_test = X.loc[train.index], X.loc[test.index]
room = [c for c in X if c.startswith("room_type_")]
models = {"baseline (median)": (DummyRegressor(strategy="median"), ["accommodates"]),
          "guests": (LinearRegression(), ["accommodates"]),
          "+ room type": (LinearRegression(), ["accommodates"] + room),
          "+ distance": (LinearRegression(), ["accommodates", "km_to_centre"] + room),
          "+ district": (LinearRegression(), list(X.columns))}
rows = []
for name, (model, cols) in models.items():
    model.fit(X_train[cols], train["log_price"])
    pred = np.exp(model.predict(X_test[cols]))
    rows.append({"model": name, "MAE": mean_absolute_error(test["price"], pred),
                 "median AE": np.median(np.abs(test["price"] - pred)),
                 "RMSE": root_mean_squared_error(test["price"], pred),
                 "R2 (log)": r2_score(test["log_price"], np.log(pred))})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
final = models["+ district"][0]
fitted = final.predict(X_test)
resid = test["log_price"] - fitted
fig, ax = plt.subplots(figsize=(6.5, 4), layout="constrained")
ax.scatter(fitted, resid, s=5, alpha=0.3, color="0.5")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="fitted log(price)", ylabel="residual", title="Residuals on the test listings")
plt.show()
cols = ["price", "room_type", "accommodates", "district", "minimum_nights", "number_of_reviews"]
print("largest positive residuals:")
print(test.assign(resid=resid).nlargest(3, "resid")[cols + ["resid"]].round(2))
print("largest negative residuals:")
print(test.assign(resid=resid).nsmallest(3, "resid")[cols + ["resid"]].round(2))

**Exercises.**

1. Describe the residual plot. What do the listings with the largest residuals have in common?
2. Add `bedrooms` to the model. It is missing for 27 % of listings (Session 4): try (a) dropping incomplete
   rows, (b) the imputation rule of Session 4 plus an indicator. Does the MAE improve, and on which rows is it
   computed? Why must the comparison use the same test rows?
3. Why would adding `estimated_revenue_l365d` improve the test score and still be wrong?

## Block 2 · Model complexity and robust regression

### 2.1 Training versus test error for increasing complexity

First on toy data (20 points from a cosine curve), then on the price model.

In [ ]:
rng = np.random.default_rng(0)
x_tr = np.sort(rng.uniform(0, 1, 20)).reshape(-1, 1)
y_tr = np.cos(1.5 * np.pi * x_tr.ravel()) + rng.normal(0, 0.15, 20)
x_te = rng.uniform(0, 1, 1000).reshape(-1, 1)
y_te = np.cos(1.5 * np.pi * x_te.ravel()) + rng.normal(0, 0.15, 1000)
rows = []
for degree in range(1, 16):
    m = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_tr, y_tr)
    rows.append({"degree": degree, "train RMSE": root_mean_squared_error(y_tr, m.predict(x_tr)),
                 "test RMSE": root_mean_squared_error(y_te, m.predict(x_te))})
curve = pd.DataFrame(rows).set_index("degree")
ax = curve.plot(logy=True, marker="o", figsize=(6, 3.5), title="Toy data: training vs test error")
ax.set_ylabel("RMSE (log scale)")
plt.show()

In [ ]:
features = ["accommodates", "km_to_centre"]
rows = []
for degree in [1, 2, 3, 4, 5, 6, 8]:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())
    m.fit(train[features], train["log_price"])
    rows.append({"degree": degree,
                 "train RMSE": root_mean_squared_error(train["log_price"], m.predict(train[features])),
                 "test RMSE": root_mean_squared_error(test["log_price"], m.predict(test[features]))})
pd.DataFrame(rows).set_index("degree").round(3)

**Exercise.** Repeat the experiment with only 60 training listings (`train.sample(60, random_state=0)`). At
which degree does the test error start to rise, and where does it explode? What does this say about flexibility
relative to the amount of data? Then try 500 training listings.

In [ ]:
# Exercise: complexity with 60 training listings
# small = train.sample(60, random_state=0)
# ...

### 2.2 Least squares versus robust regression: a host-pricing view

Fit the price in **euros** (not log) on guests and distance, so the extreme asking prices act fully, as gross
errors in y.

In [ ]:
print(train.nlargest(5, "price")[["price", "room_type", "accommodates", "km_to_centre"]].round(1))
formula = "price ~ accommodates + km_to_centre"
fits = {"OLS (mean)": smf.ols(formula, train).fit(),
        "OLS, prices < 1,000": smf.ols(formula, train[train["price"] < 1000]).fit(),
        "median (q=0.5)": smf.quantreg(formula, train).fit(q=0.5),
        "q=0.9": smf.quantreg(formula, train).fit(q=0.9)}
rows = []
for name, fit in fits.items():
    pred = fit.predict(test)
    rows.append({"model": name, "intercept": fit.params["Intercept"], "per guest": fit.params["accommodates"],
                 "per km": fit.params["km_to_centre"], "test MAE": mean_absolute_error(test["price"], pred),
                 "share of test prices below": np.mean(test["price"] <= pred)})
huber = HuberRegressor(max_iter=1000).fit(train[features], train["price"])
pred = huber.predict(test[features])
rows.append({"model": "Huber", "intercept": huber.intercept_, "per guest": huber.coef_[0], "per km": huber.coef_[1],
             "test MAE": mean_absolute_error(test["price"], pred),
             "share of test prices below": np.mean(test["price"] <= pred)})
pd.DataFrame(rows).set_index("model").round(2)

In [ ]:
grid = pd.DataFrame({"accommodates": 4, "km_to_centre": np.linspace(0, 20, 50)})
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
four = short[short["accommodates"] == 4]
ax.scatter(four["km_to_centre"], four["price"], s=5, alpha=0.3, color="0.6", label="listings for 4 guests")
for name, colour, ls in [("median (q=0.5)", "#009E73", "-"), ("q=0.9", "#0072B2", "-"), ("OLS (mean)", "#D55E00", "--")]:
    ax.plot(grid["km_to_centre"], fits[name].predict(grid), color=colour, ls=ls, lw=2, label=name)
ax.set(ylim=(0, 700), xlabel="km to Alexanderplatz", ylabel="EUR per night",
       title="Listings for four guests: mean, median and 90th percentile")
ax.legend(frameon=False, fontsize=8)
plt.show()

**Exercise.** (a) How much do the extreme prices move the least-squares distance effect, and how much the
median and Huber lines? (b) A new host has a flat for four guests 3 km from Alexanderplatz. What would you
suggest as a typical price, and what do the top 10 % of comparable listings charge? (c) Add room type to the
quantile models: what changes for private rooms? (d) Why is it still better to correct or remove confirmed
errors (Session 4) than to rely on a robust method alone?

## Block 3 · Logistic regression: Telco churn

The IBM Telco sample (7,043 customers) is downloaded from GitHub (about 1 MB). The module description asks for
churn; this block is unchanged from the plan.

In [ ]:
url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # 11 blanks -> NaN
telco["churn"] = (telco["Churn"] == "Yes").astype(int)
print(telco.shape, "churn rate:", round(telco["churn"].mean(), 3))
telco.groupby("Contract")["churn"].agg(["mean", "size"]).round(3)

In [ ]:
tr, te = train_test_split(telco, test_size=0.25, stratify=telco["churn"], random_state=0)
formula = ("churn ~ tenure + MonthlyCharges + SeniorCitizen + C(Contract) + C(InternetService)"
           " + C(PaymentMethod) + C(PaperlessBilling)")
logit = smf.logit(formula, data=tr).fit(disp=0)
coef = pd.DataFrame({"coef (log-odds)": logit.params, "odds ratio": np.exp(logit.params),
                     "OR low": np.exp(logit.conf_int()[0]), "OR high": np.exp(logit.conf_int()[1]),
                     "p": logit.pvalues})
coef.round(3)

**Exercise.** Interpret three coefficients as odds ratios (for example two-year contract, fibre optic, tenure per 12 months: `np.exp(12 * coef)`). Then translate one of them into probabilities for two typical customers with the cell below.

In [ ]:
typical = pd.DataFrame({
    "tenure": [3, 3], "MonthlyCharges": [80, 80], "SeniorCitizen": [0, 0],
    "Contract": ["Month-to-month", "Two year"], "InternetService": ["Fiber optic", "Fiber optic"],
    "PaymentMethod": ["Electronic check", "Electronic check"], "PaperlessBilling": ["Yes", "Yes"],
})
typical.assign(p_churn=logit.predict(typical).round(3))

### The same model in scikit-learn

In [ ]:
numeric = ["tenure", "MonthlyCharges", "SeniorCitizen"]
categorical = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]
pre = ColumnTransformer([("num", "passthrough", numeric),
                         ("cat", OneHotEncoder(drop="first"), categorical)])
clf = make_pipeline(pre, LogisticRegression(C=np.inf, max_iter=5000)).fit(tr, tr["churn"])
sk_coef = pd.Series(clf[-1].coef_[0], index=clf[0].get_feature_names_out())
sk_coef.round(3)

### A first look at predicted classes

In [ ]:
p = clf.predict_proba(te)[:, 1]
print("majority baseline accuracy:", round(1 - te["churn"].mean(), 3))
rows = []
for threshold in [0.5, 0.4, 0.3, 0.2]:
    pred = (p > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(te["churn"], pred).ravel()
    rows.append({"threshold": threshold, "accuracy": (pred == te["churn"]).mean(),
                 "TP (churners found)": tp, "FN (missed)": fn, "FP (false alarms)": fp, "TN": tn})
pd.DataFrame(rows).set_index("threshold").round(3)

**Exercise.** A retention call costs €10; a lost customer costs €200 of margin. Which threshold minimises the total cost on the test set (cost = 10 × (TP + FP) + 200 × FN)? Session 8 returns to this question with ROC curves.